ItemCF的实现代码，根据《推荐系统实践》这本书的思路，结合常用的数据科学计算包完成</br>
不用数据科学包的代码参考:https://github.com/Magic-Bubble/RecommendSystemPractice/blob/master/Chapter2/%E5%9F%BA%E4%BA%8E%E7%89%A9%E5%93%81%E7%9A%84%E5%8D%8F%E5%90%8C%E8%BF%87%E6%BB%A4%E7%AE%97%E6%B3%95.ipynb </br>

### 读取数据以及预处理

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
header = ['user_id', 'movie_id', 'ratings', 'timestamp']
ratings_df = pd.read_csv('ratings.dat',
                         sep='::',
                         header=None,
                         engine='python',
                         names=header,
                         usecols=[0,1,2])
#usecols 只读取前两列，本次实验用不到除了ratings.dat前两列以外的其他数据

In [3]:
ratings_df.head()

,user_id,movie_id,ratings
0,1,1193,5
1,1,661,3
2,1,914,3
3,1,3408,4
4,1,2355,5


In [4]:
# 分割，注意stratify参数必须加上，因为该数据是有序的，在1/8处直接画一条线的这种分割方式会导致无法评估算法
ratings_df_train,ratings_df_test = train_test_split(ratings_df,test_size = 0.125,stratify=ratings_df['user_id'],random_state=1)

In [5]:
# 划分后给训练集按照user_id排序(因为之前的划分过程中被打乱了)
ratings_df_train.sort_values(by='user_id', inplace=True, ascending=True)
print(ratings_df_train.shape)
ratings_df_train

(875182, 3)


,user_id,movie_id,ratings
47,1,1207,4
1,1,661,3
16,1,2687,3
21,1,720,3
19,1,2797,4
...,...,...,...
1000062,6040,1251,5
1000153,6040,2384,4
1000059,6040,1249,4
999897,6040,3068,3


In [6]:
# 同样的，给测试集排序
ratings_df_test.sort_values(by='user_id', inplace=True, ascending=True)
print(ratings_df_test.shape)
ratings_df_test

(125027, 3)


,user_id,movie_id,ratings
12,1,2398,4
4,1,2355,5
37,1,1022,5
38,1,2762,4
51,1,608,4
...,...,...,...
1000015,6040,357,3
1000066,6040,1254,5
1000008,6040,1962,3
1000079,6040,1283,4


In [7]:
# 查看数据集
ratings_df_train

,user_id,movie_id,ratings
47,1,1207,4
1,1,661,3
16,1,2687,3
21,1,720,3
19,1,2797,4
...,...,...,...
1000062,6040,1251,5
1000153,6040,2384,4
1000059,6040,1249,4
999897,6040,3068,3


### 建立用户-物品索引
表示用户对物品感兴趣的列表

In [8]:
# 将数据按照'user_id'和'movie_id'进行透视，创建用户-物品矩阵，user_item_matrix作矩阵用于今后推荐
user_item_matrix = ratings_df_train.pivot(index='user_id', columns='movie_id', values='ratings').fillna(0)
user_item_matrix

movie_id,1,2,3,4,5,6,7,8,9,10,...,3943,3944,3945,3946,3947,3948,3949,3950,3951,3952
user_id,,,,,,,,,,,,,,,,,,,,,
1,5.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6036,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6037,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6038,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


### 建立物品-物品索引

In [9]:
# 计算物品相似度矩阵
item_similarity_matrix = cosine_similarity(user_item_matrix.T)
item_similarity_df = pd.DataFrame(item_similarity_matrix, index=user_item_matrix.columns, columns=user_item_matrix.columns)

In [10]:
item_similarity_df

movie_id,1,2,3,4,5,6,7,8,9,10,...,3943,3944,3945,3946,3947,3948,3949,3950,3951,3952
movie_id,,,,,,,,,,,,,,,,,,,,,
1,1.000000,0.333967,0.229123,0.158550,0.214411,0.307014,0.259948,0.108810,0.093327,0.313214,...,0.088091,0.019604,0.073770,0.071607,0.043274,0.273778,0.165120,0.092458,0.039637,0.165109
2,0.333967,1.000000,0.220081,0.127614,0.220836,0.211031,0.232230,0.155940,0.139811,0.323126,...,0.057068,0.010009,0.063403,0.079209,0.069116,0.186104,0.121766,0.086395,0.024679,0.113652
3,0.229123,0.220081,1.000000,0.153549,0.270230,0.177698,0.241023,0.053688,0.130650,0.202401,...,0.035057,0.070160,0.046477,0.068716,0.045638,0.176243,0.093080,0.060080,0.011692,0.094214
4,0.158550,0.127614,0.153549,1.000000,0.246808,0.115854,0.193420,0.036302,0.054237,0.103544,...,0.038736,0.000000,0.002724,0.010741,0.015171,0.102599,0.068695,0.016254,0.014621,0.070926
5,0.214411,0.220836,0.270230,0.246808,1.000000,0.137666,0.268946,0.044853,0.120909,0.202486,...,0.024030,0.090015,0.050313,0.028571,0.006671,0.150561,0.089496,0.051883,0.005000,0.103456
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3948,0.273778,0.186104,0.176243,0.102599,0.150561,0.204957,0.168364,0.069535,0.081880,0.205881,...,0.171354,0.049345,0.089572,0.166163,0.081432,1.000000,0.306179,0.153413,0.113225,0.319055
3949,0.165120,0.121766,0.093080,0.068695,0.089496,0.175904,0.087697,0.055632,0.084772,0.107782,...,0.235378,0.066513,0.072977,0.159240,0.106246,0.306179,1.000000,0.269450,0.182886,0.303797
3950,0.092458,0.086395,0.060080,0.016254,0.051883,0.090736,0.047974,0.050190,0.005381,0.089073,...,0.108589,0.202189,0.069332,0.132677,0.105951,0.153413,0.269450,1.000000,0.189300,0.185793


#### 使用上述表格，可以直接查找两部电影的相似度

In [11]:
# 用这种方式可以直接验证两部电影的相似度
# 这里验证的是588号电影Aladdin和364号电影Lion King
print(item_similarity_df[588][364])

# 这里验证的是588号电影Aladdin和364号电影Toy Story
print(item_similarity_df[588][1])

0.5676349086253888
0.5311441251003869


### 两个离线计算表格已经准备就绪，现在可以做推荐了

In [12]:
# 设定要推荐的用户
user_id = 1

recommend_num = 10

# 获取该用户看过的电影
watched_items = user_item_matrix.loc[user_id][user_item_matrix.loc[user_id].astype(bool)].index
print(watched_items)

Index([   1,   48,  150,  260,  527,  531,  588,  594,  595,  661,  720,  745,
        783,  914,  919,  938, 1028, 1029, 1035, 1097, 1193, 1207, 1246, 1270,
       1287, 1545, 1566, 1721, 1836, 1907, 1961, 1962, 2018, 2028, 2294, 2321,
       2340, 2687, 2692, 2791, 2797, 2918, 3105, 3114, 3186, 3408],
      dtype='int64', name='movie_id')


In [13]:
similar_movies_num_k = 80

In [14]:
# 针对每一个用户看过的电影，通过“物品 -> 物品”的索引，找到top-k相似物品
top_k_similar_items_dict = {}

for item in watched_items:
    if item in item_similarity_df.index:
        similar_items = item_similarity_df.loc[item].sort_values(ascending=False)[1:similar_movies_num_k+1]
        top_k_similar_items_dict[item] = similar_items

top_k_similar_items_dict

# 如用户看过的1号电影，根据上列物品-物品索引，可以知道最相似的电影有3144号，1265号等等

{1: movie_id
 3114    0.552638
 1265    0.537379
 588     0.531144
 2355    0.516883
 1270    0.490126
           ...   
 587     0.399165
 32      0.399040
 2997    0.398909
 1259    0.398611
 1278    0.398551
 Name: 1, Length: 80, dtype: float64,
 48: movie_id
 783     0.456068
 595     0.436213
 1688    0.417552
 588     0.397226
 364     0.397119
           ...   
 1569    0.246027
 2746    0.245860
 362     0.245787
 736     0.245534
 239     0.244216
 Name: 48, Length: 80, dtype: float64,
 150: movie_id
 1704    0.505320
 2268    0.474058
 1682    0.472194
 318     0.463508
 1393    0.459842
           ...   
 260     0.358043
 1968    0.357504
 3253    0.356996
 1962    0.355560
 2890    0.355404
 Name: 150, Length: 80, dtype: float64,
 260: movie_id
 1196    0.693559
 1210    0.645882
 1198    0.630641
 1240    0.598796
 1214    0.598042
           ...   
 316     0.432070
 3175    0.431373
 2918    0.431191
 1573    0.430809
 3703    0.430434
 Name: 260, Length: 80, dtype: flo

In [15]:
movie_scores = {}

# 计算每个相似电影的总相似度，这里只是单纯简单地相加，实际上这样的做法需要改进，结合课件ItemCF的内容，说说你的看法
for movie, similar_items in top_k_similar_items_dict.items():
    for similar_movie, similarity_score in similar_items.items():
        if similar_movie not in movie_scores:
            movie_scores[similar_movie] = 0
        movie_scores[similar_movie] = movie_scores[similar_movie] + similarity_score * user_item_matrix.loc[user_id,movie]

# 对推荐电影按照总相似度进行排序
recommendations = sorted(movie_scores.items(), key=lambda x: x[1], reverse=True)
recommendations

[(1196, np.float64(58.14925595676199)),
 (1097, np.float64(56.2650093573387)),
 (1270, np.float64(54.15909071983632)),
 (1, np.float64(53.52689316073251)),
 (1265, np.float64(52.497596689914644)),
 (1198, np.float64(51.414838494877394)),
 (260, np.float64(48.72120733538936)),
 (1210, np.float64(47.33033760597222)),
 (2797, np.float64(44.897388021205494)),
 (2987, np.float64(43.137615048983285)),
 (919, np.float64(42.02910204095146)),
 (1197, np.float64(41.044886550557585)),
 (592, np.float64(40.618734013460866)),
 (593, np.float64(40.33598515544499)),
 (318, np.float64(39.827341958323856)),
 (2716, np.float64(39.6878425642731)),
 (588, np.float64(38.31121610391188)),
 (356, np.float64(38.28290409649106)),
 (296, np.float64(38.14295004076712)),
 (364, np.float64(37.82143164468238)),
 (1580, np.float64(37.78016136659865)),
 (2081, np.float64(37.6946233462043)),
 (2174, np.float64(37.36975287346132)),
 (1073, np.float64(36.98072733139939)),
 (608, np.float64(36.88914415021277)),
 (595, np

In [16]:
# 过滤掉目标用户已经观看过的电影，不作推荐，比如用户看过1号电影，就把1号电影过滤掉了
filtered_recommended_movies = [(movie_id, score) for movie_id, score in recommendations if movie_id not in watched_items]
filtered_recommended_movies

[(1196, np.float64(58.14925595676199)),
 (1265, np.float64(52.497596689914644)),
 (1198, np.float64(51.414838494877394)),
 (1210, np.float64(47.33033760597222)),
 (2987, np.float64(43.137615048983285)),
 (1197, np.float64(41.044886550557585)),
 (592, np.float64(40.618734013460866)),
 (593, np.float64(40.33598515544499)),
 (318, np.float64(39.827341958323856)),
 (2716, np.float64(39.6878425642731)),
 (356, np.float64(38.28290409649106)),
 (296, np.float64(38.14295004076712)),
 (364, np.float64(37.82143164468238)),
 (1580, np.float64(37.78016136659865)),
 (2081, np.float64(37.6946233462043)),
 (2174, np.float64(37.36975287346132)),
 (1073, np.float64(36.98072733139939)),
 (608, np.float64(36.88914415021277)),
 (1259, np.float64(34.23940992579588)),
 (1307, np.float64(34.21514255734086)),
 (2571, np.float64(33.85356364303985)),
 (1291, np.float64(33.56873042877438)),
 (2858, np.float64(33.35951939052473)),
 (480, np.float64(33.33811232348077)),
 (457, np.float64(32.37111630279746)),
 (102

In [17]:
# 裁减前10的作品
filtered_recommended_movies = filtered_recommended_movies[:10]
filtered_recommended_movies

[(1196, np.float64(58.14925595676199)),
 (1265, np.float64(52.497596689914644)),
 (1198, np.float64(51.414838494877394)),
 (1210, np.float64(47.33033760597222)),
 (2987, np.float64(43.137615048983285)),
 (1197, np.float64(41.044886550557585)),
 (592, np.float64(40.618734013460866)),
 (593, np.float64(40.33598515544499)),
 (318, np.float64(39.827341958323856)),
 (2716, np.float64(39.6878425642731))]

In [18]:
# 把ID筛选出来
recommended_result = [movie_id for movie_id, _ in filtered_recommended_movies]
recommended_result

[1196, 1265, 1198, 1210, 2987, 1197, 592, 593, 318, 2716]

In [19]:
# 用户1在测试集里评价过的电影（在这个实验中因为被划为测试集，所以当成是他没看过，但喜欢的电影）
test_data_user_liked = ratings_df_test[ratings_df_test['user_id'] == user_id]
real_result = test_data_user_liked["movie_id"]
real_result

12    2398
4     2355
37    1022
38    2762
51     608
5     1197
7     2804
Name: movie_id, dtype: int64

In [20]:
# 取交集用于算指标，即推荐中用户喜欢的百分比
intersection_result = list(set(real_result) & set(recommended_result))
intersection_result 

[1197]

In [21]:
# 算出精确度
precision = len(intersection_result) / len(recommended_result)
precision

0.1

### 将代码打包成函数，并使用函数测试

In [22]:
# 为了评测代码，将获取推荐的过程写成函数（用户的相似度矩阵不用再浪费时间计算）
# 函数输入用户ID，相似度矩阵，用户-物品矩阵，要找多少个相似的人做统计，推荐前多少部电影，输出为推荐的电影

def get_user_recommendations(target_user, item_similairty_df, user_item_matrix, similar_item_K, N):

    # 获取该用户看过的电影
    watched_items = user_item_matrix.loc[user_id][user_item_matrix.loc[user_id].astype(bool)].index

    # 针对每一个电影，通过“物品 -> 物品”的索引，找到top-k相似物品，这里的成品只能是一个字典，没法做成一个二维表
    top_k_similar_items_dict = {}

    for item in watched_items:
        if item in item_similarity_df.index:
            similar_items = item_similarity_df.loc[item].sort_values(ascending=False)[1:similar_movies_num_k+1]
            top_k_similar_items_dict[item] = similar_items

    movie_scores = {}

    # 计算每个用户看过电影的总相似度
    for movie, similar_items in top_k_similar_items_dict.items():
        for similar_movie, similarity_score in similar_items.items():
            if similar_movie not in movie_scores:
                movie_scores[similar_movie] = 0
            movie_scores[similar_movie] = movie_scores[similar_movie] + similarity_score * user_item_matrix.loc[target_user,movie]  

    # 对推荐电影按照总相似度进行排序
    recommendations = sorted(movie_scores.items(), key=lambda x: x[1], reverse=True)

    # 过滤用户看过的电影
    filtered_recommended_movies = [(movie_id, score) for movie_id, score in recommendations if movie_id not in watched_items]

    filtered_recommended_movies = filtered_recommended_movies[:10]
    recommended_result = [movie_id for movie_id, _ in filtered_recommended_movies]


    # 取统计分的前N个作推荐
    return recommended_result

In [23]:
user_51_ratings = user_item_matrix.loc[51,1337]
user_51_ratings

np.float64(4.0)

In [24]:
# 测试get_user_recommendations函数是否可以使用
recommendations = get_user_recommendations(51, item_similarity_df, user_item_matrix, 80, 10)
recommendations

[364, 1265, 1580, 1196, 2571, 2355, 1210, 480, 1198, 377]

### 对1~6040用户都进行推荐，并使用测试集的数据来验证

In [25]:
# 注意：按照书中的算法，在云环境中跑了300~400秒，还是可以接受的时间

all_recommendations_num = 0
precise_recommendations_num = 0

similar_movies_num_K = 10
recommend_num = 5

for user_id in set(ratings_df_test["user_id"]):

    # 在测试集中找到用户曾今看过的电影，用于作评估
    test_data_user_liked = ratings_df_test[ratings_df_test['user_id'] == user_id]
    real_result = test_data_user_liked["movie_id"]

    recommendations = get_user_recommendations(user_id, item_similarity_df, user_item_matrix, similar_movies_num_K, recommend_num)

    # 推荐的物品是用户喜欢的，intersection：交集
    intersection_result = list(set(real_result) & set(recommendations))
    
    # 统计精确推荐（推荐的刚好是用户喜欢的） 做分子 这个变量就是混淆矩阵中的TP
    precise_recommendations_num += len(intersection_result)

    # 统计推荐总数 做分母 是TP + FP
    all_recommendations_num += len(recommendations)

# 求出精确度
precision = precise_recommendations_num / all_recommendations_num

In [26]:
print(precision)

0.20652317880794702


In [27]:
len(ratings_df_test)

125027

In [28]:
# 求出混淆矩阵中的各个指标，用于后续验证
TP = precise_recommendations_num
FP = all_recommendations_num - TP

# FN：预测错了的负样本，可以通过测试集的长度 - TP得到（详情可以查看PPT中的混淆矩阵部分）
FN =  len(ratings_df_test) - TP

print("TP: " + str(TP))
print("FP: " + str(FP))
print("FN: " + str(FN))

TP: 12474
FP: 47926
FN: 112553


In [29]:
precision = TP / (TP+FP)
recall = TP/ (TP+FN)

f1_score = 2 * (precision * recall) / (precision + recall)

In [30]:
print("推荐的物品中用户喜欢的数量：" + str(precise_recommendations_num))
print("推荐的物品总数：" + str(all_recommendations_num))
print("精确度：" + str(precision))
print("召回率：" + str(recall))
print("F1-score：" + str(f1_score))

推荐的物品中用户喜欢的数量：12474
推荐的物品总数：60400
精确度：0.20652317880794702
召回率：0.0997704495828901
F1-score：0.13454351308061932
